<div style="text-align: center;">

# COSC 3P96 - Gradient Descent as Search for Facial Attribute Prediction

### Lauren Corbeil | Student ID: 7698699 | Email: wf22dy@brocku.ca
### Holly Young | Student ID: 7492895 | Email: hy21zt@brocku.ca
### Skyler Serwa | Student ID: 1111111 | Email: ss22tp@brocku.ca

### Linear and logistic regression implemented using gradient descent to predict facial attributes, with experiments analyzing learning rates, convergence, and loss behavior.

In [ ]:
# use the requirements.txt

import os
from torchvision.datasets import CelebA

# 1.

<div style="text-align: center;">

## Data Loading and Cleaning

<div style="text-align: left;">

- Download the CelebA dataset using torchvision
- Read and parse the attribute file to extract attribute names and data samples
- Select a single attribute as the target variable
- Extract labels for the selected attribute and convert values from {-1, 1} to {0, 1}
- Use a subset of the data and assign placeholder features for each sample

In [ ]:
def download_dataset(download_path = "CelebA_dataset"):

    print("if unable to download dataset from the google drive, download manually (google drive hit max daily downloads. keep same file structure as described in the code)")

    # CelebA(root = download_path, split = "train", download = True)
    # print("The dataset was downloaded.")

def load_data(file_path, attribute_name, max_samples = 5000):
    X = []
    y = []

    with open(file_path, 'r') as file:
        data = file.readlines()

    attribute_names = data[1].split()
    attribute_index = attribute_names.index(attribute_name)

    image_data = data[2:]

    for i, d in enumerate(image_data):
        if i >= max_samples:
            break
        parts = d.split()

        # The first col has the image name, we dont need it right now
        label = int(parts[attribute_index + 1])

        # Change -1 to 0
        if label == -1:
            label = 0

        y.append(label)

        X.append([1])

    return X, y

# Use the code we just made

path = "CelebA_dataset"

download_dataset(path)

attribute_file = os.path.join(path, "celeba", "list_attr_celeba.txt")

X, y = load_data(attribute_file, "Smiling")

print("Sample images: ", len(X))
print("First 10 labels:", y[:10])


if unable to download dataset from the google drive, download manually (google drive hit max daily downloads. keep same file structure as described in the code)
Sample images:  5000
First 10 labels: [1, 1, 0, 0, 0, 0, 0, 0, 1, 0]


In [ ]:
## Linear Regression Model ##

# use the requirements.txt

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import CelebA

def load_data(file_path, image_path, attribute_name, max_samples = 5000):
    X = []
    y = []

    with open(file_path, 'r') as file:
        data = file.readlines()

    attribute_names = data[1].split()
    attribute_index = attribute_names.index(attribute_name)

    image_data = data[2:]

    for i, d in enumerate(image_data):
        if i >= max_samples:
            break
        parts = d.split()

        label = int(parts[attribute_index + 1])
        image_name = parts[0]

        # Change -1 to 0
        if label == -1:
            label = 0

        # Reisizing and recoloring images for analysis.
        img = os.path.join(image_path, image_name)
        img = cv2.imread(img)
        img = cv2.resize(img, (64, 64))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        y.append(label)
        X.append(img)

    return X, y

def train_model(b, w, learning_rate, epochs, n):
    # Gradient descent loop.
    for i in range(epochs):
        y_pred = b + X @ w
        error = y_pred - y

        b -= learning_rate * (1/n) * np.sum(error)
        w -= learning_rate * (1/n) * (X.T @ error)

        if i % 2 == 0:
            mse = (1/n) * np.sum((error) ** 2)
            print(f"Training Iteration {i} MSE = {mse}")
            iterations.append(i)
            graphpoints.append(mse)
    return b, w

def confusion_matrix(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[TN, FP], [FN, TP]])

# Sorting out file paths and loading data.
print("Loading data...")

attribute_path = "CelebA_dataset"
attribute_file = os.path.join(attribute_path, "celeba", "list_attr_celeba.txt")

image_path = 'CelebA_images'
image_files = os.path.join(image_path, "img_align_celeba")

X, y = load_data(attribute_file, image_files, "Smiling")

# Converting data to numpy arrays and normalizing image pixels.
X = np.array(X) / 255.0
X = X.reshape(len(X), -1)
y = np.array(y)

# Initializing the training parameters and training the model.
b = 0
w = np.zeros(X.shape[1])
learning_rate = 0.0001
epochs = 25
n = X.shape[0]
iterations = []
graphpoints = []

b, w = train_model(b, w, learning_rate, epochs, n)

# Displaying the convergence plot.
plt.plot(iterations, graphpoints)
plt.title(f"Convergence Plot")
plt.xlabel("Training Iteration")
plt.ylabel("Mean Squared Error")
plt.show()

# Creating the confusion matrix.
y_pred = ((b + X @ w) >= 0.5).astype(int)
cm = confusion_matrix(y, y_pred)

# Displaying the confusion matrix.
plt.imshow(cm, cmap = 'Blues')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.xticks([0, 1], ["Not Smiling", "Smiling"])
plt.yticks([0, 1], ["Not Smiling", "Smiling"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha = 'center', va = 'center')

plt.colorbar()
plt.show()